<a href="https://colab.research.google.com/github/SandeepKisku24/lipur/blob/main/Lipur_Music_upload_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Music extractor for Lipur

In [ ]:
!pip install -q yt-dlp langgraph langchain-google-genai pandas requests pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.8 MB/s eta 0:00:00


It's recommended to install a JavaScript runtime like `deno` for `yt-dlp` to function optimally and avoid potential missing formats. Run the following cell to install `deno`.

In [ ]:
!curl -fsSL https://deno.land/x/install/install.sh | sh
# Add deno to PATH for current session
import os
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.deno/bin")
print("Deno installed and added to PATH.")

######################################################################## 100.0%
Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Installed dx alias, if this conflicts with an existing command, you can remove it with `rm $(which dx)` and choose a new name with `dx --install-alias <new-name>`
Deno was installed successfully to /root/.deno/bin/deno
sh: 109: cannot open /dev/tty: No such device or address
Deno installed and added to PATH.


In [ ]:
import pandas as pd

In [170]:
songs = pd.read_excel("/content/Lipur List.xlsx")
songs.columns = songs.columns.str.strip() # Clean up column names
print(songs)

                                                 url
0          https://m.youtube.com/watch?v=K_e93C6U6bg
1          https://m.youtube.com/watch?v=6NbHR4wPrOE
2          https://m.youtube.com/watch?v=lo3V-V0cUKI
3          https://m.youtube.com/watch?v=Q26zhiNTD2g
4          https://m.youtube.com/watch?v=JuJqutrSN0U
5          https://m.youtube.com/watch?v=q3av6FZZCNc
6          https://m.youtube.com/watch?v=6k-_GrB5rtg
7          https://m.youtube.com/watch?v=ulF7kYDMiWE
8          https://m.youtube.com/watch?v=4UUoXigUZQI
9          https://m.youtube.com/watch?v=Zx71zhmQM-Y
10         https://m.youtube.com/watch?v=h5jfmhSesf8
11         https://m.youtube.com/watch?v=6NbHR4wPrOE
12         https://m.youtube.com/watch?v=GoPyn5m9uIs
13         https://m.youtube.com/watch?v=5S4GnAZeI7s
14         https://m.youtube.com/watch?v=lB_eX179NYE
15         https://m.youtube.com/watch?v=HL_hnq-71Aw
16         https://m.youtube.com/watch?v=0Gjl4fTl6kg
17         https://m.youtube.com/watch?v=5SUKp

In [171]:
song_list = pd.DataFrame(songs)
print(song_list.iloc[0])

url    https://m.youtube.com/watch?v=K_e93C6U6bg
Name: 0, dtype: object


In [169]:
first_url = song_list.iloc[0]['url']
print(f"Processing URL: {first_url}")

Processing URL: https://m.youtube.com/watch?v=K_e93C6U6bg&list=RDK_e93C6U6bg&start_radio=1


In [ ]:
import yt_dlp

def get_video_info(url):
    ydl_opts = {
        'quiet': True,
        'simulate': True, # Only simulate, do not download
        'format': 'bestaudio/best', # Get best audio format info
        'extract_flat': True, # Do not extract playlists
        'force_generic_extractor': False,
        'cookiefile': '/content/cookies.txt', # Added to handle YouTube login/bot detection
        'remote_components': ['ejs:github'] # Add this to enable remote JS challenge solvers
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False) # download=False to just get info
        return info

video_info = get_video_info(first_url)
if video_info:
    print(f"Title: {video_info.get('title')}")
    print(f"Uploader: {video_info.get('uploader')}")
    print(f"Thumbnail URL: {video_info.get('thumbnail')}")
else:
    print(f"Could not retrieve information for {first_url}")

Title: ESEL KURI || NEW HIT SANTHALI VIDEO SONG || TOM MURMU || 2018©
Uploader: Tom Murmu
Thumbnail URL: https://i.ytimg.com/vi/SWIUff-1FYM/maxresdefault.jpg


This code snippet uses `yt-dlp` to extract metadata from the YouTube URL without actually downloading the video. We can use this information to populate fields like `title`, `coverUrl`, and `artists`. Next, we can integrate this into a loop to process all URLs and then focus on downloading the MP3s and generating genres.

In [ ]:
import yt_dlp
import os

# Create a temporary folder if it doesn't exist
os.makedirs("./tmp_downloads", exist_ok=True)

def get_video_info_and_download(url):
    ydl_opts = {
        'quiet': True,
        'format': 'bestaudio/best',
        'extract_flat': False, # Changed to False to ensure we get full description & date
        'cookiefile': '/content/cookies.txt',
        'remote_components': ['ejs:github'],

        # --- MISSING PIECES ADDED HERE ---
        'outtmpl': './tmp_downloads/temp_track', # Where to save the file
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }]
    }

    # Changed download=False to download=True
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return info

# Test it
first_url = "https://www.youtube.com/watch?v=SWIUff-1FYM" # Replace with your test link
video_info = get_video_info_and_download(first_url)

if video_info:
    print("--- METADATA EXTRACTED ---")
    print(f"Title: {video_info.get('title')}")
    print(f"Thumbnail URL: {video_info.get('thumbnail')}")

    # Extract the Year (yt-dlp returns YYYYMMDD, so we slice the first 4 characters)
    raw_date = video_info.get('upload_date')
    print(f"Created Year: {raw_date[:4] if raw_date else '2026'}")

    # Get the description for the AI
    desc = video_info.get('description')
    print(f"Description (First 100 chars): {desc[:100] if desc else 'None'}...")

    print("\n--- FILE STATUS ---")
    print("File downloaded to: ./tmp_downloads/temp_track.mp3")
else:
    print(f"Could not retrieve information for {first_url}")

--- METADATA EXTRACTED ---
Title: ESEL KURI || NEW HIT SANTHALI VIDEO SONG || TOM MURMU || 2018©
Thumbnail URL: https://i.ytimg.com/vi/SWIUff-1FYM/maxresdefault.jpg
Created Year: 2018
Description (First 100 chars): Here we come up with yet another Music Video of our album "ESEL KURI". 
We have a special competitio...

--- FILE STATUS ---
File downloaded to: ./tmp_downloads/temp_track.mp3


### Step 1: Define Song Metadata Structure with Pydantic

To ensure consistency and define the mandatory fields, let's create a Pydantic model for our song metadata. This will help us structure the data as it moves through our LangGraph workflow.

In [ ]:
from pydantic import BaseModel, Field, HttpUrl
from typing import Optional, List

class SongMetadata(BaseModel):
    url: HttpUrl = Field(..., description="Original YouTube or SoundCloud URL of the song.")
    title: str = Field(..., description="Cleaned title of the song.")
    coverUrl: HttpUrl = Field(..., description="URL of the song's cover image.")
    genre: str = Field(..., description="Genre of the song (LLM-generated).")
    created_year: str = Field(..., description="Year the song was created or uploaded (default if not found).")
    uploadUser: str = Field("Agent", description="User responsible for uploading the song.")
    file_path: Optional[str] = Field(None, description="Local path to the downloaded MP3 file.")
    artists: List[str] = Field(default_factory=list, description="List of artists associated with the song.")
    song_id: Optional[str] = Field(None, description="ID of the song in the music application after upload.")
    request_id: Optional[str] = Field(None, description="Unique ID for the upload request.")
    description: Optional[str] = Field(None, description="Description of the song, used for LLM processing.")


# Example usage:
# song_data = SongMetadata(url="https://www.youtube.com/watch?v=SWIUff-1FYM", title="ESEL KURI", coverUrl="https://example.com/cover.jpg", genre="Santhali", created_year="2018", artists=["Tom Murmu"])
# print(song_data.model_dump_json(indent=2))

### Step 2: Outline the LangGraph Workflow

Now that we have a structured way to handle our song metadata, let's outline the different 'nodes' or 'agents' we'll need in our LangGraph workflow to accomplish all the tasks you've described. This will help us break down the problem into manageable components.

Here's a proposed workflow:

1.  **`ExtractMetadataNode`**: Takes a song URL, uses `yt-dlp` (or similar for SoundCloud) to extract raw metadata like title, uploader, thumbnail, and upload date. It will output a `SongMetadata` object, albeit with some fields still raw or empty.
2.  **`CleanTitleNode`**: Takes the `SongMetadata` object, uses an LLM to clean and purify the `title`, removing extraneous phrases like 'new song', 'mp3', etc.
3.  **`GenerateGenreNode`**: Takes the `SongMetadata` object and possibly the description, uses an LLM to infer and assign a `genre`.
4.  **`CheckDuplicateNode`**: Calls your backend API (`https://lipur-backend.onrender.com/songs`) to check if a song with a similar title and artist already exists to prevent duplicates.
5.  **`DownloadMP3Node`**: If no duplicate is found, it downloads the MP3 file using `yt-dlp` and renames it using the cleaned title, updating the `file_path` in the `SongMetadata` object.
6.  **`UploadToAPINode`**: Uploads the downloaded MP3 and the `SongMetadata` to your backend API, obtaining a `song_id`.
7.  **`UpdateExcelNode`**: Writes the final status, `song_id`, `request_id`, and other relevant details back to your Excel sheet (`Lipur List.xlsx`).

We'll use LangGraph to orchestrate these nodes, allowing for conditional logic (e.g., skip download/upload if a duplicate is found). We'll also need to manage the state (the `SongMetadata` object) as it passes between these nodes.

### Step 3: Initialize LangGraph and Define Graph State

Let's set up the basic LangGraph structure and define the graph state, which will be our `SongMetadata` object.

In [ ]:
from langgraph.graph import StateGraph, END

# The state of our graph will be the SongMetadata object
# We'll also include a 'status' field to track the processing outcome.
class GraphState(BaseModel):
    song_metadata: SongMetadata
    status: str = Field("pending", description="Current status of the song processing (pending, downloaded, uploaded, failed).")
    error_message: Optional[str] = Field(None, description="Error message if processing fails.")
    original_excel_index: Optional[int] = Field(None, description="Original row index in the Excel sheet for updating purposes.")

# Initialize the StateGraph
workflow = StateGraph(GraphState)

print("LangGraph workflow initialized with GraphState.")

LangGraph workflow initialized with GraphState.


### Step 4: Implement `ExtractMetadataNode`

This node will leverage `yt-dlp` to get the initial metadata from the YouTube URL. It will populate the `SongMetadata` object with the raw title, cover URL, and a preliminary artist from the uploader. The `created_year` will also be extracted here. The `genre` and cleaned `title` will be handled by subsequent LLM-based nodes.

In [ ]:
import yt_dlp
from typing import Dict, Any

def extract_metadata_node(state: GraphState) -> GraphState:
    print("--- Entering ExtractMetadataNode ---")
    song_metadata = state.song_metadata
    url = str(song_metadata.url)

    ydl_opts = {
        'quiet': True,
        'simulate': True, # Only simulate, do not download MP3 yet
        'format': 'bestaudio/best',
        'extract_flat': False, # Changed to False to ensure full description and date
        'cookiefile': '/content/cookies.txt',
        'remote_components': ['ejs:github'],
        'sleep_interval': 5, # Add a delay of 5 seconds between requests
        'max_sleep_interval': 10 # Max delay of 10 seconds
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False) # Only extract info

        if info:
            # Populate SongMetadata object
            song_metadata.title = info.get('title', 'Unknown Title')
            song_metadata.coverUrl = HttpUrl(info.get('thumbnail', '')) if info.get('thumbnail') else HttpUrl("http://example.com/default_cover.jpg") # Provide a default if None

            # Extract uploader as initial artist
            uploader = info.get('uploader')
            if uploader:
                song_metadata.artists = [uploader]

            # Extract Created Year from upload_date (YYYYMMDD)
            upload_date = info.get('upload_date')
            if upload_date and len(upload_date) >= 4:
                song_metadata.created_year = upload_date[:4]
            else:
                song_metadata.created_year = 'Unknown'

            # Store description for potential LLM use later
            song_metadata.description = info.get('description', '')

            state.song_metadata = song_metadata
            state.status = "metadata_extracted"
            print(f"Metadata extracted for: {song_metadata.title}")
        else:
            state.status = "failed"
            state.error_message = f"Could not retrieve information for {url}"
            print(f"Failed to extract metadata for {url}")

    except yt_dlp.DownloadError as e:
        state.status = "failed"
        state.error_message = f"yt-dlp DownloadError: {e}"
        print(f"yt-dlp DownloadError: {e}")
    except Exception as e:
        state.status = "failed"
        state.error_message = f"An unexpected error occurred: {e}"
        print(f"An unexpected error occurred: {e}")

    return state


# Add the node to the workflow
# workflow.add_node("extract_metadata", extract_metadata_node) # Moved to graph compilation cell
print("ExtractMetadataNode defined.")

ExtractMetadataNode defined.


### Step 5: Implement `ProcessMetadataNode` with LLM

This node will use the Gemini LLM to:
1.  **Clean the song title**: Remove any extraneous information like 'NEW HIT', 'MP3', years, or copyright notices.
2.  **Extract Artists**: Identify key artists from the title or description.
3.  **Infer Genre**: Determine a suitable genre for the song based on its title, uploader, and description.

The LLM will be instructed to return its output in a JSON format for structured updates to our `SongMetadata` object.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from google.colab import userdata

# Configure Gemini API
GOOGLE_API_KEY=userdata.get('GEMINI_API_KEY')
llm = ChatGoogleGenerativeAI(model="models/gemini-flash-lite-latest", google_api_key=GOOGLE_API_KEY)

# Define the prompt for the LLM
metadata_processing_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert music metadata processor. Your task is to clean song titles, identify artists, and infer genres from provided song information like Christian as well if any ,current genre are: Pop,Country, Classical, Traditional,Christian Note: Appears as christian ,AI Song)). Always output a JSON object with 'cleaned_title', 'extracted_artists' (list of strings), and 'inferred_genre' (single string)."),
    ("human", "Process the following song details:\n\nTitle: {raw_title}\nUploader: {uploader}\nDescription: {description}\n\nExtract the cleanest possible title, a list of primary artists, and an appropriate genre. Prioritize artist names from the 'uploader' if available and reasonable. If no specific genre is clear, provide a general one like 'World Music', 'Pop', 'Folk', etc.\n\nJSON Output:")
])

# Create the LLM chain
metadata_processing_chain = metadata_processing_prompt | llm | JsonOutputParser()

def process_metadata_node(state: GraphState) -> GraphState:
    print("--- Entering ProcessMetadataNode (LLM) ---")
    song_metadata = state.song_metadata

    try:
        # Prepare input for the LLM
        raw_title = song_metadata.title
        uploader = song_metadata.artists[0] if song_metadata.artists else "Unknown Artist"
        description = song_metadata.description

        llm_response = metadata_processing_chain.invoke({
            "raw_title": raw_title,
            "uploader": uploader,
            "description": description
        })

        # Update SongMetadata with LLM's output
        song_metadata.title = llm_response.get("cleaned_title", raw_title)
        # Ensure artists list from LLM is used, or default to existing uploader if LLM doesn't provide
        if llm_response.get("extracted_artists"):
            song_metadata.artists = llm_response["extracted_artists"]
        song_metadata.genre = llm_response.get("inferred_genre", "Unknown Genre")

        state.song_metadata = song_metadata
        state.status = "metadata_processed_llm"
        print(f"LLM processed metadata for: {song_metadata.title}")

    except Exception as e:
        state.status = "failed"
        state.error_message = f"LLM processing error: {e}"
        print(f"LLM processing error: {e}")

    return state

# Add the node to the workflow
# workflow.add_node("process_metadata_llm", process_metadata_node) # Moved to graph compilation cell
print("ProcessMetadataNode (LLM) defined.")

ProcessMetadataNode (LLM) defined.


In [ ]:
import google.generativeai as genai
from google.colab import userdata # Import userdata to get the key

# Configure genai with the API key
GOOGLE_API_KEY=userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("--- Testing LLM with a sample prompt ---")

sample_raw_title = "Crazy Frog - Axel F (Official Video)"
sample_uploader = "Crazy Frog Official"
sample_description = "The official music video for Crazy Frog - Axel F."

# Assuming metadata_processing_chain is already defined and uses the LLM
try:
    print("Available Gemini models that support generateContent:")
    for m in genai.list_models():
        if "generateContent" in m.supported_generation_methods:
            print(m.name)

    print("\n--- Testing metadata_processing_chain ---")
    # Re-initialize the LLM chain to use the new model for testing
    llm_test = ChatGoogleGenerativeAI(model="models/gemini-flash-lite-latest", google_api_key=GOOGLE_API_KEY)
    metadata_processing_chain_test = metadata_processing_prompt | llm_test | JsonOutputParser()

    test_llm_response = metadata_processing_chain_test.invoke({
        "raw_title": sample_raw_title,
        "uploader": sample_uploader,
        "description": sample_description
    })
    print("LLM Test Response for metadata_processing_chain:")
    print(test_llm_response)
except Exception as e:
    print(f"Error testing LLM: {e}")

--- Testing LLM with a sample prompt ---
Available Gemini models that support generateContent:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-

### Step 6: Implement `UploadToAPINode`

This node will send the processed `SongMetadata` and the actual MP3 file (from `file_path`) to your Go API endpoint (`https://lipur-backend.onrender.com/songs`). It will then extract the `songId` from the API's response and update the `SongMetadata` object.

In [ ]:
import requests
import json
import os

def upload_to_api_node(state: GraphState) -> GraphState:
    print("--- Entering UploadToAPINode ---")
    song_metadata = state.song_metadata
    api_url = "https://lipur-backend.onrender.com/upload" # Corrected URL

    if not song_metadata.file_path or not os.path.exists(song_metadata.file_path):
        state.status = "failed"
        state.error_message = "MP3 file not found for upload."
        print(f"Error: {state.error_message}")
        return state

    try:
        # Prepare form data
        payload = {
            "title": song_metadata.title,
            "genre": song_metadata.genre,
            "createdYear": song_metadata.created_year,
            "uploadUser": song_metadata.uploadUser,
            "coverUrl": str(song_metadata.coverUrl),
            "description": song_metadata.description # Include description
        }

        # Add artists as a list of strings directly. Requests will handle multi-value form fields.
        # The API expects 'artists' key to be repeated for each artist, or a list of strings if JSON body.
        # For form-data, passing a list directly to 'data' or 'files' makes requests send multiple fields.
        for artist in song_metadata.artists:
            if 'artists' not in payload:
                payload['artists'] = []
            payload['artists'].append(artist)


        # Prepare the file to be sent
        # The Go API expects 'file' as the key for the uploaded file
        file_name = os.path.basename(song_metadata.file_path) # Use original filename or cleaned title
        files = {
            'file': (file_name, open(song_metadata.file_path, 'rb'), 'audio/mpeg')
        }

        # Send POST request
        print(f"Uploading {file_name} to {api_url}...")
        response = requests.post(api_url, data=payload, files=files)
        response.raise_for_status() # Raise an exception for HTTP errors

        api_response = response.json()
        song_id = api_response.get("songId")

        if song_id:
            song_metadata.song_id = song_id
            state.song_metadata = song_metadata
            state.status = "uploaded"
            print(f"Song '{song_metadata.title}' uploaded successfully with ID: {song_id}")
        else:
            state.status = "failed"
            state.error_message = f"API upload successful, but no songId returned: {api_response}"
            print(f"Error: {state.error_message}")

    except requests.exceptions.RequestException as e:
        state.status = "failed"
        state.error_message = f"API request failed: {e}"
        print(f"API request failed: {e}")
    except Exception as e:
        state.status = "failed"
        state.error_message = f"An unexpected error occurred during upload: {e}"
        print(f"An unexpected error occurred during upload: {e}")

    return state


# Add the node to the workflow
# workflow.add_node("upload_to_api", upload_to_api_node) # Moved to graph compilation cell
print("UploadToAPINode defined.")

UploadToAPINode defined.


### Step 7: Compile the LangGraph Workflow

Now, let's assemble the nodes into a complete LangGraph workflow. We'll define the entry point and the sequence of execution for the `extract_metadata`, `process_metadata_llm`, and `upload_to_api` nodes.

In [ ]:
# First, redefine the conditional function to handle 'duplicate' status
def check_status_for_continuation(state: GraphState) -> str:
    """
    Checks the current status of the GraphState to determine the next step.
    Returns 'continue_processing' if status is not 'failed' or 'duplicate',
    otherwise returns 'update_excel'.
    """
    if state.status == "failed" or state.status == "duplicate":
        print(f"Conditional check: Workflow is in '{state.status}' state. Routing to Excel update.")
        return "update_excel"
    else:
        print(f"Conditional check: Workflow status is '{state.status}'. Continuing processing.")
        return "continue_processing"

The `check_status_for_continuation` function will be used by the workflow to decide the next step after the `process_metadata_llm` node. Now, let's update the graph compilation to include this conditional logic.

### Step 8: Implement `UpdateExcelNode`

This node will be responsible for updating the `Lipur List.xlsx` file with the final status of each song's processing, including any `song_id` upon successful upload or `error_message` if a step failed. It will use the `original_excel_index` stored in the `GraphState` to identify the correct row to update.

In [ ]:
import pandas as pd

def update_excel_node(state: GraphState) -> GraphState:
    print("--- Entering UpdateExcelNode ---")
    song_metadata = state.song_metadata
    original_index = state.original_excel_index

    if original_index is None:
        print("Warning: original_excel_index is missing. Cannot update Excel.")
        return state

    try:
        # Load the Excel file
        # Use 'engine=openpyxl' for .xlsx files
        df = pd.read_excel('/content/Lipur List.xlsx', engine='openpyxl')

        # Ensure the DataFrame has 'Status' and 'Error Message' columns
        if 'Status' not in df.columns:
            df['Status'] = ''
        if 'Error Message' not in df.columns:
            df['Error Message'] = ''
        if 'Song ID' not in df.columns:
            df['Song ID'] = ''

        # Update the relevant row
        df.at[original_index, 'Status'] = state.status
        df.at[original_index, 'Error Message'] = state.error_message if state.error_message else ''
        if song_metadata.song_id:
            df.at[original_index, 'Song ID'] = song_metadata.song_id

        # Save the updated DataFrame back to the Excel file
        df.to_excel('/content/Lipur List.xlsx', index=False, engine='openpyxl')
        print(f"Excel updated for row {original_index} with status: {state.status}")

    except Exception as e:
        print(f"Error updating Excel for row {original_index}: {e}")
        # Do not change state status here, as this is the final logging step

    return state

# Add the node to the workflow
# workflow.add_node("update_excel", update_excel_node) # Moved to graph compilation cell
print("UpdateExcelNode defined.")

UpdateExcelNode defined.


Now that the `UpdateExcelNode` is defined, we need to adjust the workflow connections to ensure that both successful uploads and failures route through this node to log their final status in the Excel file.

### Step 9: Implement `CheckDuplicateNode`

This node will query the music application API to check for existing songs based on `title` and `artists`. If a duplicate is found, the workflow should skip the download and upload steps and directly update the Excel with a 'duplicate' status.

In [164]:
import requests
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from google.colab import userdata

def check_duplicate_node(state: GraphState) -> GraphState:
    print("--- Entering CheckDuplicateNode (LLM-based) ---")
    song_metadata = state.song_metadata
    api_url = "https://lipur-backend.onrender.com/songs"

    if not song_metadata.title or not song_metadata.artists:
        state.status = "failed"
        state.error_message = "Title or artists missing for duplicate check. Cannot perform check."
        print(f"Error: {state.error_message}")
        return state

    try:
        # 1. Initialize LLM for duplicate checking (inside node for freshness)
        GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
        llm = ChatGoogleGenerativeAI(model="models/gemini-flash-lite-latest", google_api_key=GOOGLE_API_KEY)

        # 2. Define LLM Prompt for duplicate checking
        duplicate_check_prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert music metadata analyst. Your task is to determine if a given new song is a duplicate of any song in a provided list of existing songs. Consider variations in titles (e.g., remixes, live versions, slight misspellings), artist names (e.g., aliases, features), and genre similarities. Output a JSON object with 'is_duplicate' (boolean) and 'duplicate_song_id' (string, the ID of the *most likely* duplicate if found, otherwise null). If no strong duplicate is found, set is_duplicate to false. Be strict but flexible in identifying duplicates."),
            ("human", "New Song Details:\nTitle: {new_title}\nArtists: {new_artists}\nGenre: {new_genre}\nDescription: {new_description}\n\nExisting Songs (Title, Artists, ID):\n{existing_songs_summary}\n\nIs this new song a duplicate? Provide JSON Output:")
        ])

        # 3. Create the LLM chain
        duplicate_check_chain = duplicate_check_prompt | llm | JsonOutputParser()

        # 4. Fetch existing songs from the API
        print(f"Fetching all songs from {api_url} to check for duplicates...")
        response = requests.get(api_url)
        response.raise_for_status() # Raise an exception for HTTP errors

        api_response_content = None
        try:
            api_response_content = response.json()
        except json.JSONDecodeError:
            print("Warning: API returned non-JSON response or empty content. Treating as no existing songs.")
            existing_songs = []

        existing_songs = []
        if isinstance(api_response_content, list):
            existing_songs = api_response_content
        elif isinstance(api_response_content, dict) and 'songs' in api_response_content:
            existing_songs = api_response_content['songs']
        else:
            print(f"Warning: API response was not a list or a dict with 'songs' key. Type: {type(api_response_content)}. Treating as empty list.")

        # Format existing songs for LLM input
        existing_songs_summary_lines = []
        if existing_songs:
            for es in existing_songs:
                title = es.get('title', 'Unknown Title')
                artists = ", ".join(es.get('artistNames', ['Unknown Artist']))
                song_id = es.get('id', 'Unknown ID')
                existing_songs_summary_lines.append(f"- Title: {title}, Artists: {artists}, ID: {song_id}")
        else:
            existing_songs_summary_lines.append("No existing songs in the database.")
        existing_songs_summary = "\n".join(existing_songs_summary_lines)

        # Prepare input for the LLM
        llm_input = {
            "new_title": song_metadata.title,
            "new_artists": ", ".join(song_metadata.artists) if song_metadata.artists else "Unknown Artist",
            "new_genre": song_metadata.genre if song_metadata.genre else "Unknown Genre",
            "new_description": song_metadata.description if song_metadata.description else "No description provided",
            "existing_songs_summary": existing_songs_summary
        }

        # 5. Invoke the LLM chain
        print("Invoking LLM for duplicate check...")
        llm_response = duplicate_check_chain.invoke(llm_input)
        print(f"LLM response for duplicate check: {llm_response}")

        # 6. Parse LLM response and update state
        is_duplicate = llm_response.get("is_duplicate", False)
        duplicate_song_id = llm_response.get("duplicate_song_id")

        if is_duplicate:
            state.status = "duplicate"
            state.error_message = f"Duplicate song found (LLM-based): '{song_metadata.title}' by '{', '.join(song_metadata.artists)}' (Duplicate ID: {duplicate_song_id if duplicate_song_id else 'Not provided by LLM'})"
            song_metadata.song_id = duplicate_song_id # Store existing song ID if provided by LLM
            state.song_metadata = song_metadata
            print(f"Duplicate found: {state.error_message}")
        else:
            state.status = "no_duplicate"
            print("No duplicate found (LLM-based). Proceeding.")

    except requests.exceptions.RequestException as e:
        state.status = "failed"
        state.error_message = f"API request failed during duplicate check: {e}"
        print(f"API request failed: {e}")
    except Exception as e:
        state.status = "failed"
        state.error_message = f"An unexpected error occurred during duplicate check: {e}"
        print(f"An unexpected error occurred: {e}")

    return state

print("CheckDuplicateNode defined (LLM-based).")

CheckDuplicateNode defined (LLM-based).


In [174]:
workflow = StateGraph(GraphState)

# --- NEW DownloadMP3Node definition (kept here for now, but could be in its own cell) ---
import yt_dlp
import os

def download_mp3_node(state: GraphState) -> GraphState:
    print("--- Entering DownloadMP3Node ---")
    song_metadata = state.song_metadata
    url = str(song_metadata.url)
    cleaned_title = song_metadata.title # Use the cleaned title from LLM

    # Create a temporary folder if it doesn't exist
    download_dir = "./tmp_downloads"
    os.makedirs(download_dir, exist_ok=True)

    # Sanitize title for filename
    safe_title = "".join(c for c in cleaned_title if c.isalnum() or c in (' ', '.', '_', '-')).rstrip()
    if not safe_title: # Fallback if title becomes empty after sanitization
        safe_title = "temp_audio_file"
    output_template = os.path.join(download_dir, f"{safe_title}.%(ext)s")

    ydl_opts = {
        'quiet': True,
        'format': 'bestaudio/best',
        'extract_flat': False,
        'cookiefile': '/content/cookies.txt',
        'remote_components': ['ejs:github'],
        'outtmpl': output_template, # Use cleaned title for filename
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'sleep_interval': 5, # Add a delay of 5 seconds between requests
        'max_sleep_interval': 10 # Max delay of 10 seconds
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True) # Now actually download

        # Find the downloaded file path
        # yt_dlp might add format ID or other info to the filename, so we need to find it
        # A simple way is to check the directory for a file matching the sanitized title
        downloaded_file = None
        for f_name in os.listdir(download_dir):
            if f_name.startswith(safe_title) and f_name.endswith('.mp3'):
                downloaded_file = os.path.join(download_dir, f_name)
                break

        if downloaded_file and os.path.exists(downloaded_file):
            song_metadata.file_path = downloaded_file
            state.song_metadata = song_metadata
            state.status = "mp3_downloaded"
            print(f"MP3 downloaded to: {downloaded_file}")
        else:
            state.status = "failed"
            state.error_message = f"MP3 download failed or file not found for {url}"
            print(f"MP3 download failed for {url}")

    except yt_dlp.DownloadError as e:
        state.status = "failed"
        state.error_message = f"yt-dlp DownloadError during MP3 download: {e}"
        print(f"yt-dlp DownloadError: {e}")
    except Exception as e:
        state.status = "failed"
        state.error_message = f"An unexpected error occurred during MP3 download: {e}"
        print(f"An unexpected error occurred: {e}")

    return state
# --- END DownloadMP3Node definition ---


# Add all nodes to the workflow (ensuring latest function definitions are used)
workflow.add_node("extract_metadata", extract_metadata_node)
workflow.add_node("process_metadata_llm", process_metadata_node)
workflow.add_node("check_duplicate", check_duplicate_node)
workflow.add_node("download_mp3", download_mp3_node) # download_mp3_node is defined in this cell
workflow.add_node("upload_to_api", upload_to_api_node)
workflow.add_node("update_excel", update_excel_node)
print("All nodes added to workflow.")

# Add the initial edge from extract_metadata to process_metadata_llm
workflow.add_edge("extract_metadata", "process_metadata_llm")

# Conditional edges after process_metadata_llm:
# If failed, route to update_excel immediately. Otherwise, go to check_duplicate.
workflow.add_conditional_edges(
    "process_metadata_llm",
    check_status_for_continuation,
    {
        "continue_processing": "check_duplicate", # If not failed, continue to check duplicates
        "update_excel": "update_excel"           # If failed, route to update Excel
    }
)

# Conditional edges after check_duplicate:
# If not failed/duplicate, continue processing (to download_mp3). Otherwise, update Excel.
workflow.add_conditional_edges(
    "check_duplicate",
    check_status_for_continuation,
    {
        "continue_processing": "download_mp3",
        "update_excel": "update_excel"
    }
)

# After download_mp3, go to upload_to_api
workflow.add_edge("download_mp3", "upload_to_api")

# After upload, always update Excel
workflow.add_edge("upload_to_api", "update_excel")

# Both paths (failed, duplicate, or successful) end after updating Excel
workflow.add_edge("update_excel", END)

# Set the entry point
workflow.set_entry_point("extract_metadata")

# Compile the graph
app = workflow.compile()

print("LangGraph workflow re-compiled successfully with `CheckDuplicateNode`, `DownloadMP3Node` and updated conditional logic.")

All nodes added to workflow.
LangGraph workflow re-compiled successfully with `CheckDuplicateNode`, `DownloadMP3Node` and updated conditional logic.


### Step 10: Create Main Function to Trigger Workflow

Now that the LangGraph workflow is fully defined, let's create a main function that will read the song URLs from your Excel file, initialize a `GraphState` for each, and execute the workflow. This will process each song sequentially.

In [175]:
import time

def main_workflow_trigger(song_dataframe: pd.DataFrame):
    print("---[32m Starting Main Workflow Trigger [0m---")
    results = []

    for index, row in song_dataframe.iterrows():
        url = row['url']
        print(f"\nProcessing song from URL: {url} (Original Excel Index: {index})")

        initial_song_metadata = SongMetadata(
            url=HttpUrl(url),
            title="Initial Title", # Placeholder, will be updated by LLM
            coverUrl=HttpUrl("http://example.com/default_cover.jpg"), # Placeholder
            genre="Unknown", # Placeholder
            created_year="Unknown", # Placeholder
            artists=[] # Placeholder
        )

        initial_state = GraphState(
            song_metadata=initial_song_metadata,
            status="pending",
            original_excel_index=index
        )

        try:
            # Execute the workflow for the current song
            # app.invoke returns a dictionary representation of the GraphState
            final_state_dict = app.invoke(initial_state)
            # Convert the dictionary back to a GraphState object for easier access, or access dict keys directly
            final_state = GraphState(**final_state_dict) # Create a GraphState instance from the dictionary
            print(f"Workflow finished for {url}. Final Status: \u001b[1m{final_state.status}\u001b[0m")
            results.append({
                'url': url,
                'final_status': final_state.status,
                'error_message': final_state.error_message,
                'song_id': final_state.song_metadata.song_id
            })
        except Exception as e:
            print(f"An error occurred during workflow execution for {url}: \u001b[31m{e}\u001b[0m")
            # In case of an exception before a full GraphState is returned,
            # create a consistent failure entry.
            results.append({
                'url': url,
                'final_status': 'failed',
                'error_message': str(e),
                'song_id': None
            })
        time.sleep(1) # Add a small delay to avoid overwhelming APIs

    print("\n---[32m Main Workflow Trigger Finished [0m---")
    return results

# Example of how to run it (assuming 'song_list' DataFrame is already loaded)
# Uncomment the following line to execute the workflow
final_results = main_workflow_trigger(song_list)
print("\nOverall Results:")
for res in final_results:
    print(res)

--- Starting Main Workflow Trigger ---

Processing song from URL: https://m.youtube.com/watch?v=K_e93C6U6bg (Original Excel Index: 0)
--- Entering ExtractMetadataNode ---


Metadata extracted for: JISU AM SARI PRABHU || SANTALI GOSPEL SONG|| CELESTINA MURMU || 2024
--- Entering ProcessMetadataNode (LLM) ---
LLM processed metadata for: Jisu Am Sari Prabhu
Conditional check: Workflow status is 'metadata_processed_llm'. Continuing processing.
--- Entering CheckDuplicateNode (LLM-based) ---
Fetching all songs from https://lipur-backend.onrender.com/songs to check for duplicates...
Invoking LLM for duplicate check...
LLM response for duplicate check: {'is_duplicate': True, 'duplicate_song_id': '539ca620-f0f9-4a5d-9260-b64d6d4edd56'}
Duplicate found: Duplicate song found (LLM-based): 'Jisu Am Sari Prabhu' by 'Celestina Murmu' (Duplicate ID: 539ca620-f0f9-4a5d-9260-b64d6d4edd56)
Conditional check: Workflow is in 'duplicate' state. Routing to Excel update.
--- Entering UpdateExcelNode ---
Excel updated for row 0 with status: duplicate
Workflow finished for https://m.youtube.com/watch?v=K_e93C6U6bg. Final Status: duplicate

Processing song from URL: https://m.yout

Metadata extracted for: ISORAK DULAR KATHA | SANTALI VIDEO | Fr Emmanuel Murmu |Fr Ignatius Tudu| Stephan Tudu | Manju Murmu
--- Entering ProcessMetadataNode (LLM) ---
LLM processed metadata for: Isorak Dular Katha
Conditional check: Workflow status is 'metadata_processed_llm'. Continuing processing.
--- Entering CheckDuplicateNode (LLM-based) ---
Fetching all songs from https://lipur-backend.onrender.com/songs to check for duplicates...
Invoking LLM for duplicate check...
LLM response for duplicate check: {'is_duplicate': False, 'duplicate_song_id': None}
No duplicate found (LLM-based). Proceeding.
Conditional check: Workflow status is 'no_duplicate'. Continuing processing.
--- Entering DownloadMP3Node ---


MP3 downloaded to: ./tmp_downloads/Isorak Dular Katha.mp3
--- Entering UploadToAPINode ---
Uploading Isorak Dular Katha.mp3 to https://lipur-backend.onrender.com/upload...
Song 'Isorak Dular Katha' uploaded successfully with ID: d84c71f7-adc1-473d-baa8-eb07bc006b26
--- Entering UpdateExcelNode ---
Excel updated for row 1 with status: uploaded
Workflow finished for https://m.youtube.com/watch?v=6NbHR4wPrOE. Final Status: uploaded

Processing song from URL: https://m.youtube.com/watch?v=lo3V-V0cUKI (Original Excel Index: 2)
--- Entering ExtractMetadataNode ---


Metadata extracted for: Kok Tuti Tuti Te
--- Entering ProcessMetadataNode (LLM) ---
LLM processed metadata for: Kok Tuti Tuti Te
Conditional check: Workflow status is 'metadata_processed_llm'. Continuing processing.
--- Entering CheckDuplicateNode (LLM-based) ---
Fetching all songs from https://lipur-backend.onrender.com/songs to check for duplicates...
Invoking LLM for duplicate check...
LLM response for duplicate check: {'is_duplicate': False, 'duplicate_song_id': None}
No duplicate found (LLM-based). Proceeding.
Conditional check: Workflow status is 'no_duplicate'. Continuing processing.
--- Entering DownloadMP3Node ---


...

MP3 downloaded to: ./tmp_downloads/Cak Cando.mp3
--- Entering UploadToAPINode ---
Uploading Cak Cando.mp3 to https://lipur-backend.onrender.com/upload...
Song 'Cak Cando' uploaded successfully with ID: c7ca5e2a-f523-462c-8dd1-0195252897ba
--- Entering UpdateExcelNode ---
Excel updated for row 17 with status: uploaded
Workflow finished for https://m.youtube.com/watch?v=5SUKp3HsaI8. Final Status: uploaded

Processing song from URL: https://youtu.be/blKdAyQeJfU?si=4mnkjVjuSCFZrMgY (Original Excel Index: 18)
--- Entering ExtractMetadataNode ---


Metadata extracted for: Pritiya   Napam Bela Seteren
--- Entering ProcessMetadataNode (LLM) ---
LLM processed metadata for: Pritiya Napam Bela Seteren
Conditional check: Workflow status is 'metadata_processed_llm'. Continuing processing.
--- Entering CheckDuplicateNode (LLM-based) ---
Fetching all songs from https://lipur-backend.onrender.com/songs to check for duplicates...
Invoking LLM for duplicate check...
LLM response for duplicate check: {'is_duplicate': False, 'duplicate_song_id': None}
No duplicate found (LLM-based). Proceeding.
Conditional check: Workflow status is 'no_duplicate'. Continuing processing.
--- Entering DownloadMP3Node ---


MP3 downloaded to: ./tmp_downloads/Pritiya Napam Bela Seteren.mp3
--- Entering UploadToAPINode ---
Uploading Pritiya Napam Bela Seteren.mp3 to https://lipur-backend.onrender.com/upload...
Song 'Pritiya Napam Bela Seteren' uploaded successfully with ID: 0d4ebc59-c687-48d9-a058-d6ef014fcafc
--- Entering UpdateExcelNode ---
Excel updated for row 18 with status: uploaded
Workflow finished for https://youtu.be/blKdAyQeJfU?si=4mnkjVjuSCFZrMgY. Final Status: uploaded

Processing song from URL: https://youtu.be/9aDLPgfCqug?si=bBzBG5BMfyb1wJ3e (Original Excel Index: 19)
--- Entering ExtractMetadataNode ---


Metadata extracted for: Nule Dur   Suruj Mukhi
--- Entering ProcessMetadataNode (LLM) ---
LLM processed metadata for: Nule Dur Suruj Mukhi
Conditional check: Workflow status is 'metadata_processed_llm'. Continuing processing.
--- Entering CheckDuplicateNode (LLM-based) ---
Fetching all songs from https://lipur-backend.onrender.com/songs to check for duplicates...
Invoking LLM for duplicate check...
LLM response for duplicate check: {'is_duplicate': False, 'duplicate_song_id': None}
No duplicate found (LLM-based). Proceeding.
Conditional check: Workflow status is 'no_duplicate'. Continuing processing.
--- Entering DownloadMP3Node ---


MP3 downloaded to: ./tmp_downloads/Nule Dur Suruj Mukhi.mp3
--- Entering UploadToAPINode ---
Uploading Nule Dur Suruj Mukhi.mp3 to https://lipur-backend.onrender.com/upload...
Song 'Nule Dur Suruj Mukhi' uploaded successfully with ID: 34d78cea-29bf-49cc-b03a-77adf5235be1
--- Entering UpdateExcelNode ---
Excel updated for row 19 with status: uploaded
Workflow finished for https://youtu.be/9aDLPgfCqug?si=bBzBG5BMfyb1wJ3e. Final Status: uploaded

--- Main Workflow Trigger Finished ---

Overall Results:
{'url': 'https://m.youtube.com/watch?v=K_e93C6U6bg', 'final_status': 'duplicate', 'error_message': "Duplicate song found (LLM-based): 'Jisu Am Sari Prabhu' by 'Celestina Murmu' (Duplicate ID: 539ca620-f0f9-4a5d-9260-b64d6d4edd56)", 'song_id': '539ca620-f0f9-4a5d-9260-b64d6d4edd56'}
{'url': 'https://m.youtube.com/watch?v=6NbHR4wPrOE', 'final_status': 'uploaded', 'error_message': None, 'song_id': 'd84c71f7-adc1-473d-baa8-eb07bc006b26'}
{'url': 'https://m.youtube.com/watch?v=lo3V-V0cUKI', 'fin

To execute the full workflow for all songs in your `song_list` DataFrame, uncomment the last two lines in the code cell above and run it. The `update_excel_node` will ensure that the `Lipur List.xlsx` file is updated with the status and `song_id` for each song.